# Frankie NSFW Generator — ComfyUI + RealVisXL + IPAdapter FaceID

**Photoreal** SDXL (RealVisXL V4.0) with Frankie face lock.

Runtime → **Run all**.

In [ ]:
!nvidia-smi | head -15

In [ ]:
# ── 1. Install ComfyUI + IPAdapter Plus ──
import os
if not os.path.exists('/content/ComfyUI'):
    !git clone -q https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt
if not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus'):
    !git clone -q https://github.com/cubiq/ComfyUI_IPAdapter_plus /content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus
!pip install -q insightface onnxruntime-gpu opencv-python-headless
print('ComfyUI ready')

In [ ]:
# ── 2. Download RealVisXL V4.0 (photoreal SDXL, confirmed HF URL) + IPAdapter + CLIP-H + InsightFace ──
!mkdir -p /content/ComfyUI/models/checkpoints /content/ComfyUI/models/ipadapter /content/ComfyUI/models/loras /content/ComfyUI/models/clip_vision /content/ComfyUI/models/insightface/models/buffalo_l
CKPT = '/content/ComfyUI/models/checkpoints/realvisxl_v40.safetensors'
if not os.path.exists(CKPT) or os.path.getsize(CKPT) < 500_000_000:
    !rm -f {CKPT}
    !wget -q --show-progress -O {CKPT} https://huggingface.co/SG161222/RealVisXL_V4.0/resolve/main/RealVisXL_V4.0.safetensors
sz = os.path.getsize(CKPT)
assert sz > 500_000_000, f'RealVisXL download failed (size={sz}); check URL'
print(f'RealVisXL OK: {sz/1e9:.2f} GB')
# IPAdapter weights
if not os.path.exists('/content/ComfyUI/models/ipadapter/ip-adapter-faceid-plusv2_sdxl.bin'):
    !wget -q --show-progress -O /content/ComfyUI/models/ipadapter/ip-adapter-faceid-plusv2_sdxl.bin https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl.bin
    !wget -q --show-progress -O /content/ComfyUI/models/loras/ip-adapter-faceid-plusv2_sdxl_lora.safetensors https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl_lora.safetensors
# CLIP vision
if not os.path.exists('/content/ComfyUI/models/clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors'):
    !wget -q --show-progress -O /content/ComfyUI/models/clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors
# InsightFace buffalo_l
if not os.path.exists('/content/ComfyUI/models/insightface/models/buffalo_l/det_10g.onnx'):
    !wget -q -O /tmp/buffalo_l.zip https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip
    !unzip -q -o /tmp/buffalo_l.zip -d /content/ComfyUI/models/insightface/models/buffalo_l
!ls -lh /content/ComfyUI/models/checkpoints

In [ ]:
# ── 3. Frankie reference ──
import requests
os.makedirs('/content/ComfyUI/input', exist_ok=True)
with open('/content/ComfyUI/input/frankie-ref.png', 'wb') as f:
    f.write(requests.get('https://raw.githubusercontent.com/veyrapay/frankie-assets/main/frankie-ref.png').content)
print('Frankie ref loaded')

In [ ]:
# ── 4. Restart ComfyUI server clean ──
!pkill -f 'main.py.*8188' 2>/dev/null; sleep 3
import subprocess, time, requests as r
log_f = open('/content/comfy.log', 'w')
subprocess.Popen(['python', '/content/ComfyUI/main.py', '--listen', '127.0.0.1', '--port', '8188', '--disable-auto-launch'], cwd='/content/ComfyUI', stdout=log_f, stderr=subprocess.STDOUT)
for i in range(90):
    try:
        r.get('http://127.0.0.1:8188/system_stats', timeout=2)
        print(f'ComfyUI up after {i*2}s')
        break
    except Exception:
        time.sleep(2)
else:
    !tail -50 /content/comfy.log
    raise RuntimeError('ComfyUI did not start in 180s')
# Confirm IPAdapter node is registered
ni = r.get('http://127.0.0.1:8188/object_info').json()
for n in ['IPAdapterUnifiedLoaderFaceID', 'IPAdapterFaceID', 'CheckpointLoaderSimple']:
    print(f'  {n}: {"OK" if n in ni else "MISSING"}')

In [ ]:
# ── 5. Generate 8 photoreal NSFW Frankie scenes (longer timeout, error surfacing) ──
import json, uuid, time, requests as r
POS_PREFIX = 'RAW photo, photograph, photorealistic, sharp focus, 8k uhd, dslr, natural skin texture, film grain, freckles, dark wavy hair, gold hoop earrings, sleeve tattoos, fit body, '
NEGATIVE = 'cartoon, anime, drawing, painting, illustration, 3d render, cgi, deformed, bad anatomy, extra fingers, mutated hands, watermark, text, signature, blurry, low quality, ugly, plastic skin, oversaturated, airbrushed, doll'
SCENES = [
    'topless mirror selfie in bathroom, warm bathroom lighting, intimate amateur photo',
    'topless in bed in morning light, smiling, candid amateur photo',
    'wearing black lace lingerie, sitting on bed, looking at camera, sultry, bedroom lighting',
    'in a white t-shirt and panties, kitchen in the morning, holding coffee mug, candid',
    'topless on a beach at sunset, ocean behind, wet hair, golden hour amateur photo',
    'in a sheer black robe loosely open, balcony at night, city lights behind',
    'taking a phone selfie in a black string bikini, pool behind, sun-kissed skin, mirror reflection',
    'in a tight white tank top with no bra, gym mirror selfie, sweaty post-workout',
]

def build_workflow(prompt, seed):
    return {
        '1': {'class_type': 'CheckpointLoaderSimple', 'inputs': {'ckpt_name': 'realvisxl_v40.safetensors'}},
        '2': {'class_type': 'LoadImage', 'inputs': {'image': 'frankie-ref.png'}},
        '3': {'class_type': 'IPAdapterUnifiedLoaderFaceID', 'inputs': {'model': ['1', 0], 'preset': 'FACEID PLUS V2', 'lora_strength': 0.6, 'provider': 'CUDA'}},
        '4': {'class_type': 'IPAdapterFaceID', 'inputs': {'model': ['3', 0], 'ipadapter': ['3', 1], 'image': ['2', 0], 'weight': 0.9, 'weight_faceidv2': 1.0, 'weight_type': 'linear', 'combine_embeds': 'concat', 'start_at': 0.0, 'end_at': 1.0, 'embeds_scaling': 'V only'}},
        '5': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['1', 1]}},
        '6': {'class_type': 'CLIPTextEncode', 'inputs': {'text': NEGATIVE, 'clip': ['1', 1]}},
        '7': {'class_type': 'EmptyLatentImage', 'inputs': {'width': 832, 'height': 1216, 'batch_size': 1}},
        '8': {'class_type': 'KSampler', 'inputs': {'model': ['4', 0], 'positive': ['5', 0], 'negative': ['6', 0], 'latent_image': ['7', 0], 'seed': seed, 'steps': 30, 'cfg': 6.0, 'sampler_name': 'dpmpp_2m', 'scheduler': 'karras', 'denoise': 1.0}},
        '9': {'class_type': 'VAEDecode', 'inputs': {'samples': ['8', 0], 'vae': ['1', 2]}},
        '10': {'class_type': 'SaveImage', 'inputs': {'images': ['9', 0], 'filename_prefix': 'frankie_real'}},
    }

client_id = str(uuid.uuid4())
results = []
for i, scene in enumerate(SCENES, 1):
    prompt = POS_PREFIX + scene
    wf = build_workflow(prompt, 42 + i * 7)
    print(f'\n[{i}/{len(SCENES)}] {scene}')
    sub = r.post('http://127.0.0.1:8188/prompt', json={'prompt': wf, 'client_id': client_id}).json()
    if 'error' in sub:
        print('  SUBMIT ERROR:', json.dumps(sub.get('error'))[:300])
        print('  NODE ERRORS:', json.dumps(sub.get('node_errors', {}))[:600])
        continue
    pid = sub['prompt_id']
    img_path = None
    # First gen: up to 8 min (model load). Subsequent: up to 3 min.
    max_polls = 240 if i == 1 else 90
    for poll in range(max_polls):
        time.sleep(2)
        h = r.get(f'http://127.0.0.1:8188/history/{pid}').json()
        if pid in h:
            entry = h[pid]
            outs = entry.get('outputs', {}).get('10', {}).get('images', [])
            if outs:
                f = outs[0]
                img_url = f'http://127.0.0.1:8188/view?filename={f["filename"]}&subfolder={f["subfolder"]}&type={f["type"]}'
                img_path = f'/content/real_{i:02d}.png'
                with open(img_path, 'wb') as fh:
                    fh.write(r.get(img_url).content)
                print(f'  saved {img_path} (after {poll*2}s)')
                break
            # check for error status
            status = entry.get('status', {})
            if status.get('status_str') == 'error' or any(m[0] == 'execution_error' for m in status.get('messages', [])):
                print('  EXEC ERROR:', json.dumps(status)[:600])
                break
    if img_path:
        results.append((scene, img_path))
    else:
        # On timeout, dump tail of ComfyUI log
        print(f'  TIMED OUT after {max_polls*2}s')
        if i == 1:
            print('  --- ComfyUI log tail ---')
            !tail -30 /content/comfy.log
            break  # stop on first timeout to surface root cause

print(f'\nDone. {len(results)}/{len(SCENES)} succeeded.')

In [ ]:
# ── 6. Display ──
from IPython.display import display, Markdown
from PIL import Image
for scene, path in results:
    display(Markdown(f'**{scene}**'))
    display(Image.open(path).resize((512, 768)))

In [ ]:
# ── 7. Zip & download ──
!cd /content && zip -q frankie_real_batch.zip real_*.png 2>/dev/null || true
from google.colab import files
if os.path.exists('/content/frankie_real_batch.zip') and os.path.getsize('/content/frankie_real_batch.zip') > 1000:
    files.download('/content/frankie_real_batch.zip')
else:
    print('no batch zip — check cell 5 output for errors')